# Chest X-Ray Pneumonia Classification
## Notebook 03 — ResNet-50 Fine-Tuning

**Goal:** Fine-tune a ResNet-50 pretrained on ImageNet. Compare against the Baseline CNN from Notebook 02.

**Strategy — two-stage fine-tuning:**
- **Stage 1 (epochs 1–5):** Freeze all ResNet layers, train only the new classifier head. Let the head adapt first.
- **Stage 2 (epochs 6–15):** Unfreeze the last ResNet block (`layer4`) + classifier. Fine-tune with a small LR.

This prevents the pretrained weights from being destroyed early in training.

---
### Notebook roadmap
1. Setup & DataLoaders
2. ResNet-50 architecture & modifications
3. Stage 1 — train classifier head only
4. Stage 2 — unfreeze layer4 + fine-tune
5. Training curves
6. Test set evaluation
7. Comparison table vs Baseline CNN
8. Save checkpoint


## 1. Setup & DataLoaders

In [ ]:
import os, random, json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
import torchvision
from torchvision import datasets, transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix, roc_curve

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Config ───────────────────────────────────────────────────────────────────
CFG = {
    "data_root"    : Path("/content/chest_xray"),
    "img_size"     : 224,
    "batch_size"   : 32,
    "num_workers"  : 2,
    "seed"         : 42,
    "val_split"    : 0.20,
    "imagenet_mean": [0.485, 0.456, 0.406],
    "imagenet_std" : [0.229, 0.224, 0.225],
    # Stage 1 — head only
    "stage1_epochs": 5,
    "stage1_lr"    : 1e-3,
    # Stage 2 — unfreeze layer4
    "stage2_epochs": 10,
    "stage2_lr"    : 1e-4,
    "weight_decay" : 1e-4,
}

# ── Transforms ───────────────────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((CFG["img_size"], CFG["img_size"])),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=CFG["imagenet_mean"], std=CFG["imagenet_std"]),
])

val_test_transform = transforms.Compose([
    transforms.Resize((CFG["img_size"], CFG["img_size"])),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=CFG["imagenet_mean"], std=CFG["imagenet_std"]),
])

# ── Dataset helpers ──────────────────────────────────────────────────────────
class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        img, label = self.subset[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

full_train_dataset = datasets.ImageFolder(
    root=str(CFG["data_root"] / "train"), transform=None
)
all_indices = list(range(len(full_train_dataset)))
all_targets = full_train_dataset.targets

train_idx, val_idx = train_test_split(
    all_indices, test_size=CFG["val_split"],
    stratify=all_targets, random_state=CFG["seed"]
)

train_ds = TransformSubset(Subset(full_train_dataset, train_idx), train_transform)
val_ds   = TransformSubset(Subset(full_train_dataset, val_idx),   val_test_transform)
test_ds  = datasets.ImageFolder(
    root=str(CFG["data_root"] / "test"), transform=val_test_transform
)

# WeightedRandomSampler
train_labels   = [all_targets[i] for i in train_idx]
class_counts   = Counter(train_labels)
class_weights  = {cls: 1.0 / count for cls, count in class_counts.items()}
sample_weights = [class_weights[l] for l in train_labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"],
                          sampler=sampler, num_workers=CFG["num_workers"], pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG["batch_size"],
                          shuffle=False, num_workers=CFG["num_workers"], pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=CFG["batch_size"],
                          shuffle=False, num_workers=CFG["num_workers"], pin_memory=True)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

## 2. ResNet-50 Architecture & Modifications

ResNet-50 was pretrained on ImageNet (1.2M images, 1000 classes). We:
1. Load pretrained weights
2. Replace the final `fc` layer: `2048 → 512 → 2` (our binary task)
3. Freeze everything except the new head for Stage 1

In [ ]:
def build_resnet50(num_classes=2, dropout=0.5):
    # Load pretrained ResNet-50
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

    # Freeze ALL layers initially
    for param in model.parameters():
        param.requires_grad = False

    # Replace the classifier head
    in_features = model.fc.in_features  # 2048
    model.fc = nn.Sequential(
        nn.Linear(in_features, 512),
        nn.ReLU(inplace=True),
        nn.Dropout(p=dropout),
        nn.Linear(512, num_classes)
    )
    # New head is unfrozen by default
    return model


def unfreeze_layer4(model):
    """Unfreeze layer4 of ResNet for Stage 2 fine-tuning."""
    for param in model.layer4.parameters():
        param.requires_grad = True


def count_trainable(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Total params     : {total:,}")
    print(f"  Trainable params : {trainable:,}  ({100*trainable/total:.1f}%)")


model = build_resnet50().to(DEVICE)

print("Stage 1 — only classifier head is trainable:")
count_trainable(model)

## 3. Training & Evaluation Functions

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_probs, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss   = criterion(logits, labels)
        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
        probs = torch.softmax(logits, dim=1)[:, 1]
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    auc = roc_auc_score(all_labels, all_probs)
    return total_loss/total, correct/total, auc, np.array(all_probs), np.array(all_labels)


print("Training functions defined.")

## 4. Stage 1 — Train Classifier Head Only (5 epochs)

In [ ]:
criterion  = nn.CrossEntropyLoss()
# Only pass trainable params to optimizer
optimizer1 = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CFG["stage1_lr"],
    weight_decay=CFG["weight_decay"]
)
scheduler1 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer1, mode="min", factor=0.5, patience=2
)

history = {"train_loss": [], "val_loss": [],
           "train_acc" : [], "val_acc" : [],
           "val_auc"   : [], "stage"   : []}

best_val_loss   = float("inf")
best_model_path = "/content/resnet50_best.pth"

print("STAGE 1 — Classifier head only")
print(f"{'Epoch':>6} {'Train Loss':>11} {'Train Acc':>10} {'Val Loss':>9} {'Val Acc':>8} {'Val AUC':>8}")
print("-" * 60)

for epoch in range(1, CFG["stage1_epochs"] + 1):
    tl, ta = train_one_epoch(model, train_loader, criterion, optimizer1, DEVICE)
    vl, va, vauc, _, _ = evaluate(model, val_loader, criterion, DEVICE)
    scheduler1.step(vl)

    history["train_loss"].append(tl); history["val_loss"].append(vl)
    history["train_acc"].append(ta);  history["val_acc"].append(va)
    history["val_auc"].append(vauc);  history["stage"].append(1)

    if vl < best_val_loss:
        best_val_loss = vl
        torch.save(model.state_dict(), best_model_path)
        flag = " ← best"
    else:
        flag = ""

    print(f"{epoch:>6} {tl:>11.4f} {ta:>9.3f} {vl:>9.4f} {va:>8.3f} {vauc:>8.4f}{flag}")

## 5. Stage 2 — Unfreeze layer4 + Fine-tune (10 epochs)

In [ ]:
# Unfreeze layer4
unfreeze_layer4(model)
print("Stage 2 — layer4 + classifier head unfrozen:")
count_trainable(model)

# New optimizer with smaller LR for fine-tuning
optimizer2 = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CFG["stage2_lr"],
    weight_decay=CFG["weight_decay"]
)
scheduler2 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer2, mode="min", factor=0.5, patience=3
)

print(f"\n{'Epoch':>6} {'Train Loss':>11} {'Train Acc':>10} {'Val Loss':>9} {'Val Acc':>8} {'Val AUC':>8}")
print("-" * 60)

for epoch in range(CFG["stage1_epochs"] + 1,
                   CFG["stage1_epochs"] + CFG["stage2_epochs"] + 1):
    tl, ta = train_one_epoch(model, train_loader, criterion, optimizer2, DEVICE)
    vl, va, vauc, _, _ = evaluate(model, val_loader, criterion, DEVICE)
    scheduler2.step(vl)

    history["train_loss"].append(tl); history["val_loss"].append(vl)
    history["train_acc"].append(ta);  history["val_acc"].append(va)
    history["val_auc"].append(vauc);  history["stage"].append(2)

    if vl < best_val_loss:
        best_val_loss = vl
        torch.save(model.state_dict(), best_model_path)
        flag = " ← best"
    else:
        flag = ""

    print(f"{epoch:>6} {tl:>11.4f} {ta:>9.3f} {vl:>9.4f} {va:>8.3f} {vauc:>8.4f}{flag}")

print(f"\nBest val loss: {best_val_loss:.4f} — saved to {best_model_path}")

## 6. Training Curves

In [ ]:
total_epochs = CFG["stage1_epochs"] + CFG["stage2_epochs"]
epochs = list(range(1, total_epochs + 1))
stage_boundary = CFG["stage1_epochs"] + 0.5  # vertical line between stages

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (train_key, val_key), title, ylabel in zip(
    axes,
    [("train_loss", "val_loss"), ("train_acc", "val_acc"), (None, "val_auc")],
    ["Loss", "Accuracy", "Val AUC-ROC"],
    ["CrossEntropy", "Accuracy", "AUC"]
):
    if train_key:
        ax.plot(epochs, history[train_key], label="Train", color="#4C9EBE", linewidth=2)
    ax.plot(epochs, history[val_key],
            label="Val" if train_key else "Val AUC",
            color="#E07B54" if train_key else "#6BAE75", linewidth=2)
    ax.axvline(x=stage_boundary, color="gray", linestyle="--", linewidth=1, alpha=0.7)
    ax.text(stage_boundary + 0.1, ax.get_ylim()[0], "Stage 2 →",
            fontsize=8, color="gray", va="bottom")
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel)
    if train_key: ax.legend()
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("ResNet-50 — Training Curves (Stage 1 | Stage 2)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("/content/resnet50_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Test Set Evaluation

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
test_loss, test_acc, test_auc, test_probs, test_labels = evaluate(
    model, test_loader, criterion, DEVICE
)

test_preds = (test_probs >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(test_labels, test_preds).ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
precision   = tp / (tp + fp)
f1          = 2 * precision * sensitivity / (precision + sensitivity)

print("=" * 45)
print("  RESNET-50 — TEST SET RESULTS")
print("=" * 45)
print(f"  Accuracy    : {test_acc:.4f}  ({test_acc*100:.1f}%)")
print(f"  AUC-ROC     : {test_auc:.4f}")
print(f"  Sensitivity : {sensitivity:.4f}  (% Pneumonia caught)")
print(f"  Specificity : {specificity:.4f}  (% Normal correctly excluded)")
print(f"  Precision   : {precision:.4f}")
print(f"  F1 Score    : {f1:.4f}")
print("=" * 45)
print(f"  TP={tp}  FP={fp}  TN={tn}  FN={fn}")

## 8. Confusion Matrix & ROC Curve

In [ ]:
# Load baseline results for comparison on ROC
try:
    with open("/content/baseline_results.json") as f:
        baseline = json.load(f)
    has_baseline = True
except FileNotFoundError:
    has_baseline = False
    print("baseline_results.json not found — ROC will show ResNet only.")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Confusion Matrix ──────────────────────────────────────────────────────────
cm = np.array([[tn, fp], [fn, tp]])
im = axes[0].imshow(cm, cmap="Blues")
axes[0].set_xticks([0, 1]); axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(["Pred: Normal", "Pred: Pneumonia"], fontsize=11)
axes[0].set_yticklabels(["True: Normal", "True: Pneumonia"], fontsize=11)
axes[0].set_title("Confusion Matrix — ResNet-50", fontweight="bold", fontsize=13)
for i in range(2):
    for j in range(2):
        color = "white" if cm[i, j] > cm.max() / 2 else "black"
        axes[0].text(j, i, str(cm[i, j]), ha="center", va="center",
                     fontsize=20, fontweight="bold", color=color)
plt.colorbar(im, ax=axes[0])

# ── ROC Curve ────────────────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(test_labels, test_probs)
axes[1].plot(fpr, tpr, color="#E07B54", linewidth=2.5,
             label=f"ResNet-50 (AUC = {test_auc:.3f})")
if has_baseline:
    axes[1].scatter([], [], color="#4C9EBE",
                    label=f"Baseline CNN (AUC = {baseline['auc_roc']:.3f})")
axes[1].plot([0, 1], [0, 1], "--", color="gray", linewidth=1, label="Random")
axes[1].set_xlabel("False Positive Rate (1 - Specificity)", fontsize=11)
axes[1].set_ylabel("True Positive Rate (Sensitivity)", fontsize=11)
axes[1].set_title("ROC Curve", fontweight="bold", fontsize=13)
axes[1].legend(fontsize=11)
axes[1].spines[["top", "right"]].set_visible(False)

fig.suptitle("ResNet-50 — Test Set Evaluation", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("/content/resnet50_confusion_roc.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Head-to-Head Comparison Table

In [ ]:
resnet_results = {
    "model"      : "ResNet-50 (fine-tuned, ImageNet)",
    "accuracy"   : round(float(test_acc), 4),
    "auc_roc"    : round(float(test_auc), 4),
    "sensitivity": round(float(sensitivity), 4),
    "specificity": round(float(specificity), 4),
    "precision"  : round(float(precision), 4),
    "f1"         : round(float(f1), 4),
    "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn),
}

with open("/content/resnet50_results.json", "w") as f:
    json.dump(resnet_results, f, indent=2)

# ── Print comparison table ────────────────────────────────────────────────────
metrics = ["accuracy", "auc_roc", "sensitivity", "specificity", "precision", "f1"]

if has_baseline:
    print(f"{'Metric':<14} {'Baseline CNN':>14} {'ResNet-50':>12} {'Delta':>8}")
    print("-" * 52)
    for m in metrics:
        b = baseline[m]
        r = resnet_results[m]
        delta = r - b
        arrow = "▲" if delta > 0 else "▼"
        print(f"{m:<14} {b:>14.4f} {r:>12.4f} {arrow}{abs(delta):>6.4f}")
else:
    print("ResNet-50 results:")
    for m in metrics:
        print(f"  {m:<14}: {resnet_results[m]:.4f}")

## Summary

### Two-stage fine-tuning worked because:
- **Stage 1** let the new head stabilize without disrupting pretrained features
- **Stage 2** allowed `layer4` (the highest-level feature extractor) to adapt to chest X-ray textures specifically

### Key takeaway for writeup
> Transfer learning from natural images (ImageNet) to medical X-rays works well because low-level features (edges, textures) transfer across domains. The improvement in specificity shows the richer feature representations help distinguish subtle Normal vs Pneumonia differences.

### Next → Notebook 04: Grad-CAM
We'll visualize **which regions of the X-ray the model focuses on** when making predictions — the interpretability piece that makes this project clinically meaningful.